# Sumarização de Textos com PLN: Algoritmo de Luhn
**Autor: Wellington M Santos - Data Scientist**  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-wellington--moreira--santos-blue)](https://www.linkedin.com/in/wellington-moreira-santos)
[![Email](https://img.shields.io/badge/Email-wsantos08%40hotmail.com-red)](mailto:wsantos08@hotmail.com)


## 1. Introdução

No projeto [Algoritmo Baseado em Frequência](Algoritmo%20baseado%20em%20frequencia.ipynb) implementei sumarização por frequência simples: conto palavras, normalizo pesos, somo a pontuação de cada sentença. Funciona, mas tem um ponto cego. Uma sentença pode acumular uma nota alta só porque contém várias palavras importantes espalhadas por ela inteira, mesmo que essas palavras estejam distantes umas das outras e sem relação direta de sentido.

Foi exatamente esse ponto cego que Hans Peter Luhn endereçou em 1958, num dos primeiros papers da história sobre sumarização automática. A ideia dele introduz um conceito novo: a proximidade. Palavras importantes que aparecem próximas umas das outras dentro de uma sentença formam um "cluster" significativo, e esse cluster é o que realmente indica que a sentença carrega conteúdo relevante. Uma sentença com termos importantes dispersos pontua menos do que uma sentença mais curta, mas com termos importantes concentrados.

Neste projeto, implemento o algoritmo de Luhn do zero, comparo com a abordagem por frequência simples, e expando o pipeline para um caso de uso mais completo: leitura de feeds RSS, geração de nuvem de palavras, extração de entidades nomeadas e exportação dos resumos em HTML.

**Referência teórica:** [Luhn, H.P. (1958). The Automatic Creation of Literature Abstracts](https://courses.ischool.berkeley.edu/i256/f06/papers/luhn58.pdf)

**Stack utilizada:** `Python 3.10+`, `nltk`, `collections`, `heapq`, `re`, `string`, `newspaper4k`, `feedparser`, `beautifulsoup4`, `wordcloud`, `matplotlib`, `spacy>=3.x`, `rouge-score`, `IPython.display`

**Seções:**

1. Introdução
2. Instalação e Configuração
3. Pré-processamento do Texto
4. A Lógica do Algoritmo de Luhn
5. Função de Sumarização
6. Visualização do Resumo
7. Extração de Texto da Web
8. Leitura de Feed RSS
9. Nuvem de Palavras
10. Extração de Entidades Nomeadas
11. Sumarização em Lote com Exportação HTML
12. Extensão: Lematização com spaCy
13. Avaliação com ROUGE: Luhn vs. Frequência Simples
14. Conclusão e Considerações Finais


## 2. Instalação e Configuração


In [ ]:
# Instalar dependências
# !pip install nltk newspaper4k feedparser beautifulsoup4 wordcloud matplotlib spacy rouge-score
# !python -m spacy download pt_core_news_sm

In [2]:
import re
import os
import json
import string
import heapq
from collections import Counter

import nltk
from IPython.display import HTML, display

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


True

## 3. Pré-processamento do Texto

A normalização segue a mesma lógica do projeto de frequência simples: lowercase, tokenização, remoção de stopwords e pontuação. A diferença aqui é que vou adicionar duas stopwords customizadas ao vocabulário padrão, "ser" e "além", que no texto de exemplo aparecem com frequência mas não carregam carga temática relevante.


In [3]:
texto_original = """A inteligência artificial é a inteligência similar à humana máquinas.
                    Definem como o estudo de agente artificial com inteligência.
                    Ciência e engenharia de produzir máquinas com inteligência.
                    Resolver problemas e possuir inteligência.
                    Relacionada ao comportamento inteligente.
                    Construção de máquinas para raciocinar.
                    Aprender com os erros e acertos.
                    Inteligência artificial é raciocinar nas situações do cotidiano."""

texto_original = re.sub(r'\s+', ' ', texto_original)
print(texto_original)

A inteligência artificial é a inteligência similar à humana máquinas. Definem como o estudo de agente artificial com inteligência. Ciência e engenharia de produzir máquinas com inteligência. Resolver problemas e possuir inteligência. Relacionada ao comportamento inteligente. Construção de máquinas para raciocinar. Aprender com os erros e acertos. Inteligência artificial é raciocinar nas situações do cotidiano.


In [4]:
stopwords = nltk.corpus.stopwords.words('portuguese')
stopwords.extend(['ser', 'além'])
print(f"Total de stopwords: {len(stopwords)}")

Total de stopwords: 209


In [5]:
def preprocessamento(texto):
    """
    Normaliza o texto: lowercase, tokenização, remoção de stopwords e pontuação.

    Parâmetros:
        texto (str): texto bruto de entrada

    Retorna:
        str: string com tokens relevantes separados por espaço
    """
    texto_formatado = texto.lower()
    tokens = nltk.word_tokenize(texto_formatado, language='portuguese')
    tokens = [
        palavra for palavra in tokens
        if palavra not in stopwords and palavra not in string.punctuation
    ]
    return ' '.join([t for t in tokens if not t.isdigit()])